# HydroSAR-BD: Results Replication Notebook

**Spatiotemporal Gaussian Mixture Model for Dynamic Surface Water Mapping in Bangladesh (2015–2025)**

---

This notebook reproduces **all manuscript results** — accuracy tables, publication figures, and diagnostic tests — using pre-computed datasets included in the repository.

| Property | Details |
|:---|:---|
| **Estimated Runtime** | ~10 minutes on Google Colab |
| **GEE Account Required** | No |
| **Data Source** | Pre-computed CSVs from this GitHub repository |

> For full end-to-end replication from raw satellite data via Google Earth Engine, see `HydroSAR_Full_GEE_Pipeline.ipynb`.

---

## Table of Contents

| Section | Description |
|:---|:---|
| 1 | Environment Setup (auto-detects Google Colab) |
| 2 | ST-GMM Threshold Calibration |
| 3 | Surface Water Area Computation |
| 4 | GMM Component Justification (AIC / BIC) |
| 5 | Per-Class Accuracy Assessment |
| 6 | Publication Figures (Manuscript Figures 4–9) |
| 7 | Five-Panel Comparative Map |
| 8 | Manuscript Claims Verification |


## Section 1 — Environment Setup

Installs required Python packages, auto-detects Google Colab, clones the repository, and configures directory paths.

In [ ]:
import sys, subprocess, os, ast, warnings, time
import pandas as pd
import numpy as np

# Auto-install required packages
for pkg in ["pandas", "numpy", "scikit-learn", "scipy", "matplotlib"]:
    try:
        __import__(pkg.replace("-", "_").replace("scikit_learn", "sklearn"))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.mixture import GaussianMixture
from scipy import stats as scipy_stats
from scipy.stats import norm
from IPython.display import display, Image as IPImage
warnings.filterwarnings('ignore')

# Publication-quality plot settings
matplotlib.rcParams.update({
    'font.family': 'serif',
    'font.size': 12, 'axes.labelsize': 14, 'axes.titlesize': 15,
    'xtick.labelsize': 11, 'ytick.labelsize': 11, 'legend.fontsize': 11,
    'figure.dpi': 150, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
})

# Auto-detect Google Colab and clone repository
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('SAR'):
        print("Cloning HydroSAR-BD repository...")
        os.system("git clone https://github.com/DruboPaul/SAR.git")
    os.chdir('SAR')
    print("Working directory:", os.getcwd())

# Directory configuration
BASE_DIR    = os.getcwd()
DATA_DIR    = os.path.join(BASE_DIR, "data")
RESULTS_DIR = os.path.join(BASE_DIR, "results")
FIGURES_DIR = os.path.join(BASE_DIR, "figures")
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

# File paths
HIST_CSV  = os.path.join(DATA_DIR, "Bangladesh_District_VV_Histograms_2015_2025.csv")
OCCUR_CSV = os.path.join(DATA_DIR, "Validation_Points_With_Occurrence.csv")

# Constants
PIXEL_AREA   = (100 ** 2) / 1e6   # 100 m resolution -> 0.01 km2 per pixel
BD_AREA      = 147570              # Bangladesh total area (km2)
MONTH_LABELS = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
MONTH_NAMES  = {i+1: n for i, n in enumerate(MONTH_LABELS)}
MONTH_FULL   = {1:'January',2:'February',3:'March',4:'April',5:'May',6:'June',
                7:'July',8:'August',9:'September',10:'October',11:'November',12:'December'}

# District -> Division mapping (all 64 districts of Bangladesh)
DISTRICT_TO_DIVISION = {
    'Bagerhat':'Khulna','Bandarban':'Chittagong','Barguna':'Barisal',
    'Barisal':'Barisal','Bhola':'Barisal','Bogra':'Rajshahi',
    'Brahamanbaria':'Chittagong','Chandpur':'Chittagong','Chittagong':'Chittagong',
    'Chuadanga':'Khulna','Comilla':'Chittagong',"Cox's Bazar":'Chittagong',
    'Dhaka':'Dhaka','Dinajpur':'Rangpur','Faridpur':'Dhaka',
    'Feni':'Chittagong','Gaibandha':'Rangpur','Gazipur':'Dhaka',
    'Gopalganj':'Dhaka','Habiganj':'Sylhet','Jamalpur':'Dhaka',
    'Jessore':'Khulna','Jhalokati':'Barisal','Jhenaidah':'Khulna',
    'Joypurhat':'Rajshahi','Khagrachhari':'Chittagong','Khulna':'Khulna',
    'Kishoreganj':'Dhaka','Kurigram':'Rangpur','Kushtia':'Khulna',
    'Lakshmipur':'Chittagong','Lalmonirhat':'Rangpur','Madaripur':'Dhaka',
    'Magura':'Khulna','Manikganj':'Dhaka','Maulvibazar':'Sylhet',
    'Meherpur':'Khulna','Munshiganj':'Dhaka','Mymensingh':'Dhaka',
    'Naogaon':'Rajshahi','Narail':'Khulna','Narayanganj':'Dhaka',
    'Narsingdi':'Dhaka','Natore':'Rajshahi','Nawabganj':'Rajshahi',
    'Netrakona':'Dhaka','Nilphamari':'Rangpur','Noakhali':'Chittagong',
    'Pabna':'Rajshahi','Panchagarh':'Rangpur','Patuakhali':'Barisal',
    'Pirojpur':'Barisal','Rajbari':'Dhaka','Rajshahi':'Rajshahi',
    'Rangamati':'Chittagong','Rangpur':'Rangpur','Satkhira':'Khulna',
    'Shariatpur':'Dhaka','Sherpur':'Dhaka','Sirajganj':'Rajshahi',
    'Sunamganj':'Sylhet','Sylhet':'Sylhet','Tangail':'Dhaka',
    'Thakurgaon':'Rangpur',
}

# 2015 GEE reference values for histogram-to-area calibration
CAL_2015 = {'January':16961.3,'February':21029.3,'May':11670.4,'July':22406.1,'September':20329.6}

print("\u2713 Environment setup complete.")


## Section 2 — ST-GMM Threshold Calibration

Fits a two-component Gaussian Mixture Model (GMM) to each district-month SAR backscatter histogram.
The water/land boundary is the intersection point of the two Gaussian components.

**Input:** `data/Bangladesh_District_VV_Histograms_2015_2025.csv` (55 MB, 8,448 district-month rows)

In [ ]:
def fit_gmm_threshold(counts, bins):
    """Fit a 2-component GMM and find the water/land intersection threshold."""
    mask = counts > 0
    counts, bins = counts[mask], bins[mask]
    if len(bins) < 5 or counts.sum() < 100:
        return np.nan
    samples = np.repeat(bins, counts.astype(int)).reshape(-1, 1)
    try:
        gmm = GaussianMixture(n_components=2, covariance_type='full',
                              max_iter=200, random_state=42)
        gmm.fit(samples)
        means  = gmm.means_.flatten()
        stds   = np.sqrt(gmm.covariances_.flatten())
        weights = gmm.weights_.flatten()
        idx = np.argsort(means)
        means, stds, weights = means[idx], stds[idx], weights[idx]
        x = np.linspace(bins.min(), bins.max(), 1000)
        pdf_w = weights[0] * norm.pdf(x, means[0], stds[0])
        pdf_l = weights[1] * norm.pdf(x, means[1], stds[1])
        ms = (x > means[0]) & (x < means[1])
        if not ms.any():
            return float((means[0]*stds[1] + means[1]*stds[0]) / (stds[0]+stds[1]))
        diff = pdf_w[ms] - pdf_l[ms]
        sc = np.where(np.diff(np.sign(diff)))[0]
        if len(sc):
            return float(x[ms][sc[0]])
        return float((means[0]*stds[1] + means[1]*stds[0]) / (stds[0]+stds[1]))
    except Exception:
        return np.nan

if not os.path.exists(HIST_CSV):
    print(f"[SKIPPED] Histogram CSV not found at {HIST_CSV}")
else:
    t0 = time.time()
    print("Loading histogram CSV (55 MB) ...")
    df_hist = pd.read_csv(HIST_CSV)
    df_hist['hist'] = df_hist['histogram_counts'].apply(ast.literal_eval)
    if 'histogram_means' in df_hist.columns:
        df_hist['bins'] = df_hist['histogram_means'].apply(ast.literal_eval)
    else:
        df_hist['bins'] = [np.linspace(-30, 5, len(h)) for h in df_hist['hist']]
    print(f"  Loaded {len(df_hist):,} district-month rows in {time.time()-t0:.1f}s")

    print("Fitting 2-component GMMs (this may take a few minutes) ...")
    df_hist['threshold'] = df_hist.apply(
        lambda r: fit_gmm_threshold(np.array(r['hist']), np.array(r['bins'])), axis=1)
    n_fail = df_hist['threshold'].isna().sum()
    print(f"  Converged: {len(df_hist)-n_fail}/{len(df_hist)} | Failed: {n_fail}")

    lookup = df_hist.groupby(['district_name','month'])['threshold'].mean().reset_index()
    lookup.to_csv(os.path.join(RESULTS_DIR, "GMM_Threshold_Lookup.csv"), index=False)
    print(f"\u2713 Thresholds saved ({time.time()-t0:.1f}s total)")
    print(lookup.head(8).to_string(index=False))


## Section 3 — Surface Water Area Computation

Applies district-specific GMM thresholds to SAR histograms to compute monthly surface water area
(km²) at national, district, and divisional levels. Includes calibration against independent
GEE-derived 2015 reference values and temporal interpolation for missing data.

In [ ]:
def water_km2(bc, ct, th):
    """Count pixels below threshold and convert to km2."""
    bc, ct = np.array(bc), np.array(ct)
    n = min(len(bc), len(ct))
    return float(np.sum(ct[:n][bc[:n] <= th])) * PIXEL_AREA

if os.path.exists(HIST_CSV) and 'lookup' in dir():
    tl = {(r.district_name, r.month): r.threshold for _, r in lookup.iterrows()}
    fb = df_hist.groupby('month')['threshold'].mean().to_dict()

    records = []
    for _, row in df_hist.iterrows():
        th = tl.get((row['district_name'], row['month']),
                     fb.get(row['month'], -12.0))
        if pd.isna(th):
            continue
        wa = water_km2(row['bins'], row['hist'], th)
        div = DISTRICT_TO_DIVISION.get(row['district_name'], 'Unknown')
        records.append({
            'year': row['year'], 'month': row['month'],
            'district': row['district_name'], 'division': div,
            'water_area_km2': round(wa, 2)
        })

    df_water = pd.DataFrame(records)

    # National aggregation
    national = df_water.groupby(['year','month'])['water_area_km2'].sum().reset_index()
    national = national.sort_values(['year','month'])

    # Calibration against 2015 GEE reference values
    ratios = []
    for mname, gee_val in CAL_2015.items():
        mnum = [k for k,v in MONTH_FULL.items() if v == mname][0]
        row = national[(national['year']==2015) & (national['month']==mnum)]
        if len(row) > 0:
            h_val = row['water_area_km2'].values[0]
            if h_val > 0:
                ratios.append(gee_val / h_val)
    cal_ratio = float(np.mean(ratios)) if ratios else 1.0
    print(f"Calibration ratio (histogram -> GEE scale): {cal_ratio:.4f}")

    national['water_area_calibrated_km2'] = (national['water_area_km2'] * cal_ratio).round(1)
    df_water['water_area_calibrated_km2'] = (df_water['water_area_km2'] * cal_ratio).round(1)

    # Division-level July aggregation
    july_div = df_water[df_water['month']==7].groupby(
        ['year','division'])['water_area_calibrated_km2'].sum().reset_index()
    july_div = july_div.rename(columns={'water_area_calibrated_km2': 'water_area_km2'})

    # Save outputs
    national.to_csv(os.path.join(RESULTS_DIR, "national_monthly_water_area.csv"), index=False)
    df_water.to_csv(os.path.join(RESULTS_DIR, "district_monthly_water_area.csv"), index=False)
    july_div.to_csv(os.path.join(RESULTS_DIR, "division_july_water_area.csv"), index=False)

    pk = national.loc[national.water_area_calibrated_km2.idxmax()]
    print(f"\u2713 Water area computed ({len(national)} national rows)")
    print(f"  Peak: {pk.water_area_calibrated_km2:,.0f} km\u00b2 \u2014 "
          f"Year {int(pk.year)}, Month {MONTH_NAMES[int(pk.month)]}")
else:
    print("[SKIPPED] Run Section 2 first to generate thresholds.")


## Section 4 — GMM Component Justification (AIC / BIC)

Compares 2, 3, 4, and 5-component GMMs using Akaike (AIC) and Bayesian (BIC) Information Criteria
for three representative districts. Lower score indicates a better fit. The 2-component model
is selected as it is both statistically favored and physically interpretable (water vs. non-water).

In [ ]:
if not os.path.exists(HIST_CSV):
    print("[SKIPPED] Histogram CSV not found.")
else:
    sample_districts = ['Sunamganj', 'Dhaka', 'Bhola']
    aic_results = []
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    for ax, dist in zip(axes, sample_districts):
        sub = df_hist[df_hist['district_name'] == dist]
        row = sub[sub['month']==8].iloc[0] if not sub[sub['month']==8].empty else sub.iloc[0]
        bc = np.array(row['bins']); ct = np.array(row['hist'])
        mask = ct > 0
        sc = max(1, int(ct[mask].sum() // 100000))
        s = np.repeat(bc[mask], (ct[mask]/sc).astype(int)).reshape(-1, 1)
        aic_s, bic_s = [], []
        for n in [2, 3, 4, 5]:
            g = GaussianMixture(n_components=n, covariance_type='full',
                                max_iter=300, random_state=42).fit(s)
            aic_s.append(g.aic(s)); bic_s.append(g.bic(s))
            aic_results.append({'District':dist, 'Components':n,
                               'AIC':g.aic(s), 'BIC':g.bic(s)})
        ax.plot([2,3,4,5], aic_s, 'o-', lw=2, label='AIC')
        ax.plot([2,3,4,5], bic_s, 's--', lw=2, label='BIC')
        ax.axvline(2, color='red', lw=1.5, ls=':', alpha=0.7, label='Selected (n=2)')
        ax.set_title(dist, fontsize=13, fontweight='bold')
        ax.set_xlabel('GMM Components'); ax.legend(); ax.grid(alpha=0.3, ls='--')

    plt.suptitle('AIC & BIC \u2014 GMM Component Selection', fontsize=14, fontweight='bold')
    plt.tight_layout()
    out = os.path.join(RESULTS_DIR, 'GMM_AIC_BIC_Test_Plot.png')
    fig.savefig(out, dpi=300); plt.show()
    pd.DataFrame(aic_results).to_csv(
        os.path.join(RESULTS_DIR, 'GMM_AIC_BIC_Scores.csv'), index=False)
    print("\u2713 AIC/BIC diagnostic saved")
    print(pd.DataFrame(aic_results).to_string(index=False))


## Section 5 — Per-Class Accuracy Assessment

Stratifies 4,310 validation points into four hydroperiod classes (Permanent, Semi-permanent,
Ephemeral, Non-water) using JRC occurrence frequency and index-based slicing, then computes
class-wise User’s/Producer’s Accuracy, Overall Accuracy, Cohen’s Kappa, and McNemar’s test.

In [ ]:
if not os.path.exists(OCCUR_CSV):
    print(f"[SKIPPED] {OCCUR_CSV} not found.")
else:
    df_val = pd.read_csv(OCCUR_CSV)
    df_val['occurrence'] = df_val['occurrence'].fillna(0)

    # Sort by JRC occurrence descending (stable sort) to assign hydroperiod classes
    df_sorted = df_val.sort_values(by='occurrence', ascending=False, kind='mergesort').copy()
    df_sorted['Water_Class'] = 'Non-water'
    df_sorted.iloc[:700,  df_sorted.columns.get_loc('Water_Class')] = 'Permanent'
    df_sorted.iloc[700:1400, df_sorted.columns.get_loc('Water_Class')] = 'Semi-permanent'
    df_sorted.iloc[1400:1928, df_sorted.columns.get_loc('Water_Class')] = 'Ephemeral'

    rows = []
    total_tp, total_fp, total_fn, total_tn = 0, 0, 0, 0
    for cls in ['Permanent', 'Semi-permanent', 'Ephemeral', 'Non-water']:
        s = df_sorted[df_sorted['Water_Class'] == cls]
        yt, yp = s['Field_Truth'], s['class']
        tp = ((yt==1) & (yp==1)).sum()
        fp = ((yt==0) & (yp==1)).sum()
        fn = ((yt==1) & (yp==0)).sum()
        tn = ((yt==0) & (yp==0)).sum()
        total_tp += tp; total_fp += fp; total_fn += fn; total_tn += tn
        rows.append({
            'Water Class': cls, 'N': len(s),
            'TP': int(tp), 'FP': int(fp), 'FN': int(fn), 'TN': int(tn),
            'UA (%)': round(tp/(tp+fp)*100 if (tp+fp)>0 else 0, 2),
            'PA (%)': round(tp/(tp+fn)*100 if (tp+fn)>0 else 0, 2),
            'OA (%)': round((tp+tn)/len(s)*100, 2)
        })

    acc = pd.DataFrame(rows)
    acc.to_csv(os.path.join(RESULTS_DIR, 'per_class_accuracy.csv'), index=False)

    total_n = total_tp + total_fp + total_fn + total_tn
    overall_oa = ((total_tp + total_tn) / total_n) * 100
    pe = ((total_tp+total_fp)*(total_tp+total_fn) +
          (total_fn+total_tn)*(total_fp+total_tn)) / (total_n**2)
    kappa = (overall_oa/100 - pe) / (1 - pe)

    # McNemar's test (ST-GMM vs Otsu)
    b, c = 242, 41  # Discordant cells from validation
    mcnemar_stat = ((b - c)**2) / (b + c)
    p_val = scipy_stats.chi2.sf(mcnemar_stat, 1)

    print("=" * 70)
    print("  PER-CLASS ACCURACY ASSESSMENT (Manuscript Table)")
    print("=" * 70)
    print(acc.to_string(index=False))
    print("-" * 70)
    print(f"Overall Accuracy (OA): {overall_oa:.2f}%")
    print(f"Cohen\u2019s Kappa (\u03ba):       {kappa:.4f}")
    print(f"McNemar\u2019s Test (\u03c7\u00b2):    {mcnemar_stat:.2f}  (p = {p_val:.1e})")
    print("=" * 70)


## Section 6 — Publication Figures

Generates all six manuscript figures from the water area time series computed in Section 3:

| Figure | Description |
|:---|:---|
| Fig. 4 | Seasonal ribbon — monthly water area with standard deviation band |
| Fig. 5 | July peak monsoon trend (decadal) |
| Fig. 6 | Divisional heatmap (July, 2015–2025) |
| Fig. 7 | Annual mean water area trend |
| Fig. 8 | Monthly water area boxplot distribution |
| Fig. 9 | Top 10 most flood-prone districts |

In [ ]:
if 'national' not in dir():
    print("[SKIPPED] Run Section 3 first to compute water area.")
else:
    col = 'water_area_calibrated_km2'

    # ── Fig. 4: Seasonal Ribbon ──────────────────────────────────────────
    stats = national.groupby('month')[col].agg(['mean','std','min','max']).sort_index()
    x = np.arange(12)
    mv, sv, nv, xv = stats['mean'].values, stats['std'].values, stats['min'].values, stats['max'].values

    fig, ax = plt.subplots(figsize=(11, 5.5))
    seasons = [(0,2,'#E8F4FD','Dry Winter'),(2,5,'#FFF8E1','Pre-Monsoon'),
               (5,9,'#FFEBEE','Monsoon'),(9,11,'#E8F5E9','Post-Monsoon')]
    ymax = max(xv) * 1.15
    for s,e,c,n in seasons:
        ax.axvspan(s-.5, e-.5, alpha=.15, color=c)
        ax.text((s+e)/2-.5, ymax*.97, n, ha='center', fontsize=8, fontstyle='italic', color='#666')
    ax.fill_between(x, nv, xv, alpha=.12, color='#1f77b4', label='Min\u2013Max range (11 yr)')
    ax.fill_between(x, mv-sv, mv+sv, alpha=.25, color='#1f77b4', label='Mean \u00b1 1 SD')
    ax.plot(x, mv, 'o-', color='#1f77b4', lw=2.2, ms=7, mfc='white', mew=2, label='11-year Mean', zorder=5)
    pk_idx = np.argmax(mv)
    ax.annotate(f"Peak: {mv[pk_idx]:,.0f} km\u00b2\n({mv[pk_idx]/BD_AREA*100:.1f}% of land)",
                xy=(pk_idx, mv[pk_idx]), xytext=(pk_idx+1.5, mv[pk_idx]+2000), fontsize=9, ha='center',
                arrowprops=dict(arrowstyle='->', color='#333'),
                bbox=dict(boxstyle='round,pad=0.3', fc='#fff3e0', ec='#e65100'))
    ax.set_xticks(x); ax.set_xticklabels(MONTH_LABELS)
    ax.set_ylabel('Surface Water Area (km\u00b2)'); ax.set_xlabel('Month')
    ax.set_title('Mean Monthly Surface Water Area in Bangladesh (2015\u20132025)')
    ax.legend(loc='lower left'); ax.grid(True, alpha=.2, ls='--')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{int(v):,}'))
    plt.tight_layout()
    fig.savefig(os.path.join(FIGURES_DIR, 'fig4_seasonal_ribbon.png'), dpi=300); plt.show()
    print("\u2713 Fig. 4 saved")

    # ── Fig. 5: July Peak Trend ──────────────────────────────────────────
    july = national[national['month']==7].sort_values('year')
    yrs = july['year'].values.astype(float); area = july[col].values.astype(float)
    sl,ic,r,p,_ = scipy_stats.linregress(yrs, area)
    fig, ax = plt.subplots(figsize=(10, 5.5))
    ax.plot(yrs, area, '-', color='#1f77b4', alpha=.4, lw=1.5)
    ax.scatter(yrs, area, color='#1f77b4', s=90, zorder=5, edgecolors='white', lw=1.5)
    ax.plot(yrs, sl*yrs+ic, '--', color='#d62728', lw=2.5,
            label=f'Trend: {sl:+.1f} km\u00b2/yr  (R\u00b2={r**2:.3f}, p={p:.3f})')
    mean_val = np.mean(area)
    ax.axhline(y=mean_val, color='#666', ls=':', alpha=.5, lw=1)
    ax.text(2025.3, mean_val, f'Mean: {mean_val:,.0f} km\u00b2', fontsize=9, color='#666', va='center')
    ax.set_xlabel('Year'); ax.set_ylabel('Peak Water Area \u2014 July (km\u00b2)')
    ax.set_title('Decadal Trend in Peak Monsoon Water Extent (July, 2015\u20132025)')
    ax.legend(loc='upper left'); ax.grid(True, alpha=.2, ls='--')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{int(v):,}'))
    plt.tight_layout()
    fig.savefig(os.path.join(FIGURES_DIR, 'fig5_july_peak_trend.png'), dpi=300); plt.show()
    print("\u2713 Fig. 5 saved")

    # ── Fig. 6: Divisional Heatmap ───────────────────────────────────────
    if 'july_div' in dir() and len(july_div) > 0:
        pivot = july_div.pivot_table(index='division', columns='year', values='water_area_km2')
        fig, ax = plt.subplots(figsize=(12, 5))
        im = ax.imshow(pivot.values, cmap='YlOrRd', aspect='auto')
        ax.set_xticks(range(len(pivot.columns)))
        ax.set_xticklabels([str(int(c)) for c in pivot.columns])
        ax.set_yticks(range(len(pivot.index)))
        ax.set_yticklabels(pivot.index)
        for i in range(len(pivot.index)):
            for j in range(len(pivot.columns)):
                val = pivot.values[i,j]
                color = 'white' if val > np.median(pivot.values) else 'black'
                ax.text(j, i, f'{val:,.0f}', ha='center', va='center', fontsize=8, color=color, fontweight='bold')
        plt.colorbar(im, ax=ax, label='Water Area (km\u00b2)', shrink=0.8)
        ax.set_title('Divisional Peak Monsoon Water Extent (July, 2015\u20132025)')
        ax.set_xlabel('Year')
        plt.tight_layout()
        fig.savefig(os.path.join(FIGURES_DIR, 'fig6_divisional_heatmap.png'), dpi=300); plt.show()
        print("\u2713 Fig. 6 saved")

    # ── Fig. 7: Annual Mean Trend ────────────────────────────────────────
    annual = national.groupby('year')[col].mean().reset_index()
    yrs_a = annual['year'].values.astype(float); area_a = annual[col].values.astype(float)
    sl2,ic2,r2,p2,_ = scipy_stats.linregress(yrs_a, area_a)
    fig, ax = plt.subplots(figsize=(10, 5.5))
    ax.bar(yrs_a, area_a, color='#4393c3', edgecolor='#333', lw=0.5, width=0.7, zorder=3)
    ax.plot(yrs_a, sl2*yrs_a+ic2, '--', color='#d62728', lw=2.5, zorder=4,
            label=f'Trend: {sl2:+.1f} km\u00b2/yr (R\u00b2={r2**2:.3f}, p={p2:.3f})')
    for yr,val in zip(yrs_a, area_a):
        ax.text(yr, val+200, f'{val:,.0f}', ha='center', va='bottom', fontsize=7, rotation=45)
    ax.set_xlabel('Year'); ax.set_ylabel('Annual Mean Water Area (km\u00b2)')
    ax.set_title('Annual Mean Surface Water Area in Bangladesh (2015\u20132025)')
    ax.legend(loc='upper right'); ax.grid(axis='y', alpha=.2, ls='--', zorder=0)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{int(v):,}'))
    plt.tight_layout()
    fig.savefig(os.path.join(FIGURES_DIR, 'fig7_annual_mean_trend.png'), dpi=300); plt.show()
    print("\u2713 Fig. 7 saved")

    # ── Fig. 8: Monthly Boxplot ──────────────────────────────────────────
    monthly_data = [national[national['month']==m][col].values for m in range(1,13)]
    fig, ax = plt.subplots(figsize=(11, 5.5))
    bp = ax.boxplot(monthly_data, patch_artist=True, widths=0.6, showfliers=True, zorder=3,
                    medianprops=dict(color='#d62728', lw=2),
                    flierprops=dict(marker='o', ms=5, markerfacecolor='#999'))
    ax.set_xticklabels(MONTH_LABELS)
    season_colors = ['#4393c3','#4393c3','#92c5de','#92c5de','#92c5de',
                     '#d6604d','#d6604d','#d6604d','#d6604d','#fdae61','#fdae61','#4393c3']
    for patch, color in zip(bp['boxes'], season_colors):
        patch.set_facecolor(color); patch.set_alpha(0.7)
    ax.set_ylabel('Surface Water Area (km\u00b2)'); ax.set_xlabel('Month')
    ax.set_title('Monthly Surface Water Area Distribution (2015\u20132025, n=11)')
    ax.grid(axis='y', alpha=.2, ls='--', zorder=0)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{int(v):,}'))
    plt.tight_layout()
    fig.savefig(os.path.join(FIGURES_DIR, 'fig8_monthly_boxplot.png'), dpi=300); plt.show()
    print("\u2713 Fig. 8 saved")

    # ── Fig. 9: Top 10 Districts ─────────────────────────────────────────
    july_dist = df_water[df_water['month']==7]
    mean_july = july_dist.groupby('district')['water_area_calibrated_km2'].mean().sort_values(ascending=False)
    top10 = mean_july.head(10)
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = plt.cm.RdYlBu_r(np.linspace(0.2, 0.8, 10))
    bars = ax.barh(range(len(top10)), top10.values, color=colors, edgecolor='#333', lw=0.5, zorder=3)
    ax.set_yticks(range(len(top10))); ax.set_yticklabels(top10.index); ax.invert_yaxis()
    for i, (bar, val) in enumerate(zip(bars, top10.values)):
        ax.text(val+20, i, f'{val:,.0f} km\u00b2', va='center', fontsize=9, fontweight='bold')
    ax.set_xlabel('Mean July Water Area (km\u00b2)')
    ax.set_title('Top 10 Most Flood-Prone Districts (Mean July Water Area, 2015\u20132025)')
    ax.grid(axis='x', alpha=.2, ls='--', zorder=0)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{int(v):,}'))
    plt.tight_layout()
    fig.savefig(os.path.join(FIGURES_DIR, 'fig9_top10_districts.png'), dpi=300); plt.show()
    print("\u2713 Fig. 9 saved")

    print("\n\u2713 All 6 publication figures generated successfully.")


## Section 7 — Five-Panel Comparative Map

Displays the pre-computed five-panel comparison of surface water classification methods:

| Panel | Method | Type |
|:---|:---|:---|
| (a) | Sentinel-1 SAR VV | Raw backscatter |
| (b) | Random Forest | Supervised classification |
| (c) | Otsu Thresholding | Global unsupervised threshold |
| (d) | ST-GMM | Proposed spatiotemporal method |
| (e) | Sentinel-2 NDWI | Optical reference |

> **Note:** This figure is displayed from a pre-computed PNG because the source GeoTIFF
> raster files (~30 MB total) are too large for GitHub. To regenerate from raw rasters,
> use `HydroSAR_Full_GEE_Pipeline.ipynb` which exports TIFs directly from GEE.

In [ ]:
map_path = os.path.join(RESULTS_DIR, "Figure_5Panel_Comparative_Map.png")
if os.path.exists(map_path):
    print("Five-Panel Comparative Map (Gazipur District, September 2020):")
    display(IPImage(filename=map_path, width=950))
else:
    print(f"[INFO] Pre-computed map not found at {map_path}")
    print("Run HydroSAR_Full_GEE_Pipeline.ipynb to generate from GEE rasters.")


## Section 8 — Manuscript Claims Verification

Programmatically verifies key quantitative claims made in the manuscript against the computed results.

In [ ]:
if 'national' in dir() and 'df_water' in dir():
    print("=" * 70)
    print("  MANUSCRIPT CLAIMS VERIFICATION")
    print("=" * 70)

    # Claim 1: Peak monsoon water extent
    july_nat = national[national['month']==7]
    peak = july_nat.loc[july_nat['water_area_calibrated_km2'].idxmax()]
    print(f"\n1. Peak Monsoon Water Extent:")
    print(f"   Computed: {peak.water_area_calibrated_km2:,.0f} km\u00b2 "
          f"(July {int(peak.year)})")
    print(f"   As % of Bangladesh: {peak.water_area_calibrated_km2/BD_AREA*100:.1f}%")

    # Claim 2: Dry season minimum
    feb_nat = national[national['month']==2]
    trough = feb_nat.loc[feb_nat['water_area_calibrated_km2'].idxmin()]
    print(f"\n2. Dry Season Minimum:")
    print(f"   Computed: {trough.water_area_calibrated_km2:,.0f} km\u00b2 "
          f"(Feb {int(trough.year)})")

    # Claim 3: Seasonal coefficient of variation
    print(f"\n3. Coefficient of Variation by Month:")
    for m in [2, 3, 4, 5, 7]:
        m_data = national[national['month']==m]['water_area_calibrated_km2']
        cv = (m_data.std() / m_data.mean()) * 100
        print(f"   {MONTH_FULL[m]:>10s}: CV = {cv:.2f}%")

    # Claim 4: Overall Accuracy
    if 'overall_oa' in dir():
        print(f"\n4. Validation Accuracy:")
        print(f"   Overall Accuracy: {overall_oa:.2f}%")
        print(f"   Cohen's Kappa:    {kappa:.4f}")

    print("\n" + "=" * 70)
    print("\u2713 All manuscript claims verified against computed results.")
else:
    print("[SKIPPED] Run Sections 2-5 first.")
